## **Import Library**

In [1]:
import pandas as pd
import ast
import nltk
from nltk.stem import PorterStemmer
import pickle

## **Load Dataset & Download some package**

In [2]:
# Load Dataset
movies = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

In [3]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## **Preprocessing**

In [4]:
movies.shape, credits.shape

((4803, 20), (4803, 4))

### **Merge Dataset**

In [5]:
movies = movies.merge(credits, left_on = 'id', right_on = 'movie_id')
movies.shape

(4803, 24)

In [6]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='str')

In [7]:
# selecting relevant features
movies = movies[['movie_id', 'title_x', 'genres', 'keywords', 'overview', 'cast', 'crew', 'vote_count', 'vote_average']]

# rename title_x to title (just for better understanding)
movies = movies.rename(columns = {'title_x': 'title'})

In [8]:
movies.head(2)

,movie_id,title,genres,keywords,overview,cast,crew,vote_count,vote_average
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",11800,7.2
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",4500,6.9


### **Convert JSON to List py**

Because the `[genres, keywords, cast]` columns store data in `str.JSON format`, they are converted with `ast.literal_eval()` for next step preprocessing.

In [9]:
def parse_json(x):
    try:
        return [i['name'] for i in ast.literal_eval(x)]
    except:
        return []

movies['genres'] = movies['genres'].apply(parse_json)
movies['keywords'] = movies['keywords'].apply(parse_json)
movies['cast'] = movies['cast'].apply(parse_json)

### **Get 3 Top Cast Selected**
Because, some films can have more casts than others, so films with a large cast do not receive too much weight.

In [10]:
movies['cast'] = movies['cast'].apply(lambda x:x[:3] if isinstance(x, list) else [])

### **Get director name**
`Why only get the director name?` Because it only retrieves the director who best represents the film.

`Why don't you use parse_json?` Because it will return all_crew members, but we only need the director name.

In [11]:
def get_director(x):
    try: 
        for i in ast.literal_eval(x):
            if i['job'] == 'Director':
                return [i['name']]
        return []
    except:
        return []
    
movies['crew'] = movies['crew'].apply(get_director)

In [12]:
# Rename crew to director
movies = movies.rename(columns = {'crew': 'director'})

In [13]:
movies[['title', 'genres', 'keywords', 'cast', 'director']].head(2)

,title,genres,keywords,cast,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


### **NLTK for overview**

`Why just overview?` Because overview is a long sentence and offers the most benefits compared to the others.

In [14]:
ps = PorterStemmer()

def stem_text(txt):
    return ' '.join([ps.stem(word) for word in str(txt).split()])

movies['overview'] = movies['overview'].apply(stem_text)
movies['overview'].iloc[0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization.'

### **Merge Some Feature**
`features: [genres, keywords, cast, director]` 

All features are combined into a single string, which will then be vectorized.

In [15]:
def merge_list_str(x):
    return ' '.join(x['genres']) + ' ' + ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + ' '.join(x['director']) + ' ' + str(x['overview'])

movies['combined_features'] = movies.apply(merge_list_str, axis=1)
movies[['title', 'combined_features']].head(2)

,title,combined_features
0,Avatar,Action Adventure Fantasy Science Fiction cultu...
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action ocean drug abuse exot...


### **Weighted rating**
for filtered movies with unrepresentative data (low votes but high ratings).

using the `IMDB (Internet Movie Database)` formula

In [16]:
m = movies['vote_count'].quantile(0.75)
C = movies['vote_average'].mean()

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v+m)  * R) + (m / (v+m) * C)

movies['score'] = movies.apply(weighted_rating, axis=1)
movies[['title', 'vote_count', 'vote_average', 'score']].head(2)

,title,vote_count,vote_average,score
0,Avatar,11800,7.2,7.134875
1,Pirates of the Caribbean: At World's End,4500,6.9,6.786315


## **Save the data**

In [17]:
movies.to_csv('../data/preprocessed/pre_movies.csv', index=False)

In [18]:
movies_list = movies['title']
with open('../data/preprocessed//movies_list.pkl', 'wb') as f:
    pickle.dump(movies_list, f)